# Análisis de correlaciones de features

Exploramos las correlaciones entre las métricas featurizadas y el resultado binario `WL_NUM` para priorizar las señales más informativas.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')


In [ ]:
FEATURIZED_PATH = "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00d_featurized/2024-25/teamgamelogs_featurized.parquet"
TARGET_COLUMN = 'WL_NUM'


In [ ]:
df = pd.read_parquet(FEATURIZED_PATH)
if TARGET_COLUMN not in df.columns:
    raise KeyError(f'No se encontró la columna objetivo {TARGET_COLUMN!r} en el dataset featurizado.')

numeric_df = df.select_dtypes(include=[np.number]).copy()
print(f'Filas: {len(numeric_df)} | Columnas numéricas: {numeric_df.shape[1]}')


In [ ]:
corr_with_target = numeric_df.corr()[TARGET_COLUMN].drop(TARGET_COLUMN).dropna()

top_pos = corr_with_target.sort_values(ascending=False).head(10)
top_neg = corr_with_target.sort_values().head(10)
least_corr = corr_with_target.abs().sort_values().head(5)

print('Top 10 correlaciones positivas')
display(top_pos.to_frame(name='Correlación'))

print('Top 10 correlaciones negativas')
display(top_neg.to_frame(name='Correlación'))

print('5 correlaciones más débiles (en valor absoluto)')
display(least_corr.to_frame(name='|Correlación|'))


## Heatmap de las 20 features con mayor correlación absoluta

Analizamos la matriz de correlaciones entre las 20 variables más asociadas al objetivo para detectar grupos redundantes o patrones conjuntos.


In [ ]:
top20_cols = corr_with_target.abs().sort_values(ascending=False).head(20).index.tolist()
heatmap_df = numeric_df[top20_cols + [TARGET_COLUMN]].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_df, annot=False, cmap='coolwarm', center=0)
plt.title('Correlaciones (top 20 features vs WL_NUM)')
plt.tight_layout()


## Dispersión de las 5 correlaciones más fuertes

Visualizamos la relación entre `WL_NUM` y las cinco features con mayor correlación absoluta.


In [ ]:
top5_cols = corr_with_target.abs().sort_values(ascending=False).head(5).index.tolist()
fig, axes = plt.subplots(nrows=1, ncols=len(top5_cols), figsize=(4 * len(top5_cols), 4), sharey=True)
if len(top5_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, top5_cols):
    sns.scatterplot(data=numeric_df, x=col, y=TARGET_COLUMN, alpha=0.3, ax=ax)
    ax.set_title(col)
plt.suptitle('Dispersión vs WL_NUM (top 5 correlaciones absolutas)', y=1.02)
plt.tight_layout()
